# Label Drift Manual Review

This notebook creates a 50-row manual review template for LLM paraphrases. Fill `gold_label_after_review` and `notes` manually, then rerun the agreement cell.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists():
    ROOT = ROOT.parent

raw_path = ROOT / 'data' / 'augmented' / 'llm_raw_0.05_42.csv'
qc_path = ROOT / 'results' / 'tables' / 'phase6_filter_qc_0.05_42.csv'
output_path = ROOT / 'results' / 'tables' / 'manual_drift_review_template.csv'

raw = pd.read_csv(raw_path) if raw_path.exists() else pd.DataFrame()
qc = pd.read_csv(qc_path) if qc_path.exists() else pd.DataFrame()

if raw.empty:
    raise FileNotFoundError(f'Missing raw paraphrase file: {raw_path}')

sample = raw.groupby('label', group_keys=False).apply(
    lambda group: group.sample(n=min(len(group), max(1, round(50 * len(group) / len(raw)))), random_state=42)
).head(50).reset_index(drop=True)

if not qc.empty and {'source_index', 'label', 'keep', 'reasons'}.issubset(qc.columns):
    sample = sample.merge(
        qc[['source_index', 'label', 'keep', 'reasons']],
        on=['source_index', 'label'],
        how='left',
    )

review = sample.rename(columns={
    'original_sentence': 'original',
    'augmented_sentence': 'paraphrase',
    'label': 'original_label',
})
review['gold_label_after_review'] = ''
review['notes'] = ''
output_path.parent.mkdir(parents=True, exist_ok=True)
review.to_csv(output_path, index=False)
output_path, review.head()

After manual annotation, reload the CSV and compute the label preservation rate.

In [ ]:
reviewed = pd.read_csv(output_path)
filled = reviewed[reviewed['gold_label_after_review'].notna() & (reviewed['gold_label_after_review'].astype(str) != '')].copy()
if filled.empty:
    print('No manual labels filled yet.')
else:
    filled['gold_label_after_review'] = filled['gold_label_after_review'].astype(int)
    agreement = (filled['original_label'].astype(int) == filled['gold_label_after_review']).mean()
    print(f'Manual label preservation rate: {agreement:.4f} over {len(filled)} reviewed rows')